# Unified Model Building Pipeline

Single-notebook ILPD modeling workflow with one manual dataset-selection cell, legacy-style per-model sections, and code-generated summary tables.

In [19]:
!pip install imblearn

In [20]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import BaggingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.options.display.float_format = "{:.4f}".format

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate repository root from current working directory")

NOTEBOOK_PATH = ROOT / "scripts" / "05_model_building_pipeline.ipynb"
REPORTS_DIR = ROOT / "produced_reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [21]:
DATASET_NAME = "raw"  # raw | cleaned | clinically_capped | robust_scaled | gmm_augmented

if DATASET_NAME == "raw":
    DATASET_PATH = ROOT / "data" / "raw" / "ILPD.csv"
    HAS_HEADER = False
# elif DATASET_NAME == "cleaned":
#     DATASET_PATH = ROOT / "data" / "interim" / "ILPD_cleaned.csv"
#     HAS_HEADER = True
elif DATASET_NAME == "clinically_capped":
    DATASET_PATH = ROOT / "data" / "processed" / "ILPD_clinically_capped.csv"
    HAS_HEADER = True
# elif DATASET_NAME == "robust_scaled":
#     DATASET_PATH = ROOT / "data" / "processed" / "ILPD_robust_scaled_features.csv"
#     HAS_HEADER = True
elif DATASET_NAME == "gmm_augmented":
    DATASET_PATH = ROOT / "data" / "processed" / "ILPD_robust_scaled_with_gmm_noise.csv"
    HAS_HEADER = True
else:
    raise ValueError("Unsupported DATASET_NAME")

USE_IMBALANCED_LEARN = DATASET_NAME == "raw"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_SPLITS = 3
HTML_OUTPUT_STEM = f"05_model_building_pipeline_{DATASET_NAME}data"

print({
    "dataset_name": DATASET_NAME,
    "dataset_path": str(DATASET_PATH),
    "has_header": HAS_HEADER,
    "use_imbalanced_learn": USE_IMBALANCED_LEARN,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_splits": CV_SPLITS,
    "html_output": f"{HTML_OUTPUT_STEM}.html",
})


{'dataset_name': 'raw', 'dataset_path': '/Users/samyabrataroy/Downloads/Spring_Internship_2026/data/raw/ILPD.csv', 'has_header': False, 'use_imbalanced_learn': True, 'random_state': 42, 'test_size': 0.2, 'cv_splits': 3, 'html_output': '05_model_building_pipeline_rawdata.html'}


In [22]:
RAW_COLUMN_NAMES = [
    "Age",
    "Gender",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Proteins",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
    "Result",
]


def standardize_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.rename(columns={"Target": "Result", "Albumin_Globulin_Ratio": "Albumin_and_Globulin_Ratio"})

    if "Result" not in df.columns and df.shape[1] == len(RAW_COLUMN_NAMES):
        df.columns = RAW_COLUMN_NAMES

    if "Gender" in df.columns and df["Gender"].dtype == object:
        normalized = df["Gender"].astype(str).str.strip().str.lower()
        df["Gender"] = normalized.map({"male": 0, "female": 1, "m": 0, "f": 1})

    for col in df.columns:
        if col == "Result":
            continue
        if df[col].dtype == object:
            try:
                df[col] = pd.to_numeric(df[col], errors="raise")
            except (TypeError, ValueError):
                df[col], _ = pd.factorize(df[col].astype(str))

    df = df.replace([np.inf, -np.inf], np.nan)
    df["Result"] = pd.to_numeric(df["Result"], errors="coerce")
    return df


def load_selected_dataset() -> pd.DataFrame:
    if HAS_HEADER:
        df = pd.read_csv(DATASET_PATH)
    else:
        df = pd.read_csv(DATASET_PATH, header=None, names=RAW_COLUMN_NAMES)
    return standardize_schema(df)


def build_preprocessor(scale_features: bool) -> Pipeline:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_features:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps)


def maybe_apply_smote(X_train: np.ndarray, y_train: pd.Series, random_state: int, use_smote: bool) -> tuple[np.ndarray, np.ndarray]:
    if not (USE_IMBALANCED_LEARN and use_smote):
        return X_train, np.asarray(y_train)

    try:
        from imblearn.over_sampling import SMOTE
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "DATASET_NAME='raw' requires imbalanced-learn. Install project dependencies and rerun."
        ) from exc

    sampler = SMOTE(random_state=random_state)
    return sampler.fit_resample(X_train, np.asarray(y_train))


def build_balanced_bagging_estimator(random_state: int):
    if USE_IMBALANCED_LEARN:
        try:
            from imblearn.ensemble import BalancedBaggingClassifier
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError(
                "DATASET_NAME='raw' requires imbalanced-learn. Install project dependencies and rerun."
            ) from exc

        return BalancedBaggingClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=6,
                min_samples_leaf=4,
                random_state=random_state,
            ),
            n_estimators=15,
            max_samples=0.8,
            bootstrap=True,
            random_state=random_state,
            n_jobs=1,
        )

    return BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=4,
            class_weight="balanced",
            random_state=random_state,
        ),
        n_estimators=15,
        max_samples=0.8,
        bootstrap=True,
        random_state=random_state,
        n_jobs=1,
    )


def positive_class_scores(estimator: object, X_eval: np.ndarray, positive_label: int) -> np.ndarray:
    if hasattr(estimator, "predict_proba"):
        classes = list(estimator.classes_)
        return estimator.predict_proba(X_eval)[:, classes.index(positive_label)]
    return np.full(len(X_eval), np.nan)


def build_metric_row(y_true: pd.Series, y_pred: np.ndarray, y_score: np.ndarray, positive_label: int) -> dict[str, float]:
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    labels = sorted(pd.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)])))
    negative_label = next(label for label in labels if label != positive_label)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[negative_label, positive_label]).ravel()

    roc_auc = np.nan
    if np.isfinite(y_score).all():
        roc_auc = roc_auc_score((np.asarray(y_true) == positive_label).astype(int), y_score)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "roc_auc": roc_auc,
        "minority_precision": report[str(positive_label)]["precision"],
        "minority_recall": report[str(positive_label)]["recall"],
        "minority_f1": report[str(positive_label)]["f1-score"],
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def fit_model_split(
    estimator: object,
    scale_features: bool,
    use_smote: bool,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    positive_label: int,
    random_state: int,
) -> dict[str, float]:
    preprocessor = build_preprocessor(scale_features)
    X_train_tx = preprocessor.fit_transform(X_train)
    X_test_tx = preprocessor.transform(X_test)
    X_fit, y_fit = maybe_apply_smote(X_train_tx, y_train, random_state=random_state, use_smote=use_smote)

    model = clone(estimator)
    model.fit(X_fit, y_fit)
    y_pred = model.predict(X_test_tx)
    y_score = positive_class_scores(model, X_test_tx, positive_label)
    return build_metric_row(y_test, y_pred, y_score, positive_label)


def cross_validate_model(
    estimator: object,
    scale_features: bool,
    use_smote: bool,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    positive_label: int,
    random_state: int,
    cv_splits: int,
) -> pd.DataFrame:
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    rows = []
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
        fold_row = fit_model_split(
            estimator=estimator,
            scale_features=scale_features,
            use_smote=use_smote,
            X_train=X_train.iloc[train_idx],
            y_train=y_train.iloc[train_idx],
            X_test=X_train.iloc[val_idx],
            y_test=y_train.iloc[val_idx],
            positive_label=positive_label,
            random_state=random_state + fold,
        )
        fold_row["fold"] = fold
        rows.append(fold_row)
    return pd.DataFrame(rows)


def build_model_row(
    model_name: str,
    estimator: object,
    scale_features: bool,
    use_smote: bool,
) -> dict[str, float]:
    split_metrics = fit_model_split(
        estimator=estimator,
        scale_features=scale_features,
        use_smote=use_smote,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        positive_label=POSITIVE_LABEL,
        random_state=RANDOM_STATE,
    )
    cv_df = cross_validate_model(
        estimator=estimator,
        scale_features=scale_features,
        use_smote=use_smote,
        X_train=X_train,
        y_train=y_train,
        positive_label=POSITIVE_LABEL,
        random_state=RANDOM_STATE,
        cv_splits=CV_SPLITS,
    )

    return {
        "model_name": model_name,
        "estimator_class": type(estimator).__name__,
        "scale_features": scale_features,
        "used_smote": bool(USE_IMBALANCED_LEARN and use_smote),
        "used_imbalanced_learn": USE_IMBALANCED_LEARN,
        "test_accuracy": split_metrics["accuracy"],
        "test_balanced_accuracy": split_metrics["balanced_accuracy"],
        "test_f1_macro": split_metrics["f1_macro"],
        "test_f1_weighted": split_metrics["f1_weighted"],
        "roc_auc": split_metrics["roc_auc"],
        "minority_precision": split_metrics["minority_precision"],
        "minority_recall": split_metrics["minority_recall"],
        "minority_f1": split_metrics["minority_f1"],
        "cv_accuracy_mean": float(cv_df["accuracy"].mean()),
        "cv_accuracy_std": float(cv_df["accuracy"].std(ddof=0)),
        "cv_balanced_accuracy_mean": float(cv_df["balanced_accuracy"].mean()),
        "cv_f1_macro_mean": float(cv_df["f1_macro"].mean()),
        "cv_gap_abs": float(abs(cv_df["accuracy"].mean() - split_metrics["accuracy"])),
        "tn": split_metrics["tn"],
        "fp": split_metrics["fp"],
        "fn": split_metrics["fn"],
        "tp": split_metrics["tp"],
    }


In [23]:
df = load_selected_dataset()

FEATURE_COLUMNS = [col for col in df.columns if col not in {"Result", "Synthetic_Flag"}]
X = df[FEATURE_COLUMNS].copy().apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(df["Result"], errors="coerce")

valid_mask = y.notna()
X = X.loc[valid_mask].reset_index(drop=True)
y = y.loc[valid_mask].astype(int).reset_index(drop=True)

CLASS_COUNTS = y.value_counts().sort_index()
POSITIVE_LABEL = int(CLASS_COUNTS.idxmin())
NEGATIVE_LABEL = int(CLASS_COUNTS.idxmax())
SYNTHETIC_ROWS = 0
if "Synthetic_Flag" in df.columns:
    SYNTHETIC_ROWS = int(pd.to_numeric(df["Synthetic_Flag"], errors="coerce").fillna(0).sum())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

DATASET_SUMMARY = pd.DataFrame(
    [
        {
            "dataset_name": DATASET_NAME,
            "dataset_path": str(DATASET_PATH),
            "rows": int(len(X)),
            "columns": int(len(df.columns)),
            "feature_count": int(len(FEATURE_COLUMNS)),
            "class_1_count": int(CLASS_COUNTS.get(1, 0)),
            "class_2_count": int(CLASS_COUNTS.get(2, 0)),
            "minority_label": POSITIVE_LABEL,
            "majority_label": NEGATIVE_LABEL,
            "minority_count": int(CLASS_COUNTS.min()),
            "majority_count": int(CLASS_COUNTS.max()),
            "imbalance_ratio_majority_to_minority": float(CLASS_COUNTS.max() / CLASS_COUNTS.min()),
            "used_imbalanced_learn": USE_IMBALANCED_LEARN,
            "synthetic_rows": SYNTHETIC_ROWS,
            "train_rows": int(len(X_train)),
            "test_rows": int(len(X_test)),
        }
    ]
)

display(DATASET_SUMMARY)


,dataset_name,dataset_path,rows,columns,feature_count,class_1_count,class_2_count,minority_label,majority_label,minority_count,majority_count,imbalance_ratio_majority_to_minority,used_imbalanced_learn,synthetic_rows,train_rows,test_rows
0,raw,/Users/samyabrataroy/Downloads/Spring_Internsh...,583,11,10,416,167,2,1,167,416,2.4910,True,0,466,117


# Model 1: Random Forest

In [24]:
RANDOM_FOREST_RESULT = pd.DataFrame(
    [
        build_model_row(
            model_name="RandomForest",
            estimator=RandomForestClassifier(
                n_estimators=150,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
            scale_features=False,
            use_smote=True,
        )
    ]
)

display(RANDOM_FOREST_RESULT)


,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,roc_auc,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,RandomForest,RandomForestClassifier,False,True,True,0.6325,0.6107,0.5940,0.6464,0.7077,...,0.4691,0.6781,0.0096,0.6217,0.6130,0.0456,55,28,15,19


# Model 2: Optimized Logistic Regression

In [25]:
LOGISTIC_REGRESSION_RESULT = pd.DataFrame(
    [
        build_model_row(
            model_name="LogisticRegression",
            estimator=LogisticRegression(
                C=0.5,
                class_weight="balanced",
                max_iter=5000,
                solver="liblinear",
                random_state=RANDOM_STATE,
            ),
            scale_features=True,
            use_smote=False,
        )
    ]
)

display(LOGISTIC_REGRESSION_RESULT)


,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,roc_auc,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,LogisticRegression,LogisticRegression,True,False,True,0.6325,0.6976,0.6255,0.6469,0.7392,...,0.5743,0.6459,0.0192,0.7027,0.6347,0.0134,45,38,5,29


# Model 3: Stacking Classifier

In [26]:
STACKING_RESULT = pd.DataFrame(
    [
        build_model_row(
            model_name="Stacking_LR_RF",
            estimator=StackingClassifier(
                estimators=[
                    (
                        "lr",
                        LogisticRegression(
                            C=0.5,
                            class_weight="balanced",
                            max_iter=5000,
                            solver="liblinear",
                            random_state=RANDOM_STATE,
                        ),
                    ),
                    (
                        "rf",
                        RandomForestClassifier(
                            n_estimators=100,
                            min_samples_leaf=2,
                            class_weight="balanced_subsample",
                            random_state=RANDOM_STATE,
                            n_jobs=1,
                        ),
                    ),
                ],
                final_estimator=LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
                cv=3,
                n_jobs=1,
                passthrough=False,
                stack_method="predict_proba",
            ),
            scale_features=True,
            use_smote=True,
        )
    ]
)

display(STACKING_RESULT)


,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,roc_auc,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,Stacking_LR_RF,StackingClassifier,True,True,True,0.6410,0.6081,0.5962,0.6525,0.7123,...,0.4615,0.6995,0.0140,0.6525,0.6415,0.0585,57,26,16,18


# Model 4: HistGradientBoosting

In [27]:
HIST_GRADIENT_BOOSTING_RESULT = pd.DataFrame(
    [
        build_model_row(
            model_name="HistGradientBoosting",
            estimator=HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=120,
                max_depth=5,
                min_samples_leaf=20,
                random_state=RANDOM_STATE,
            ),
            scale_features=False,
            use_smote=True,
        )
    ]
)

display(HIST_GRADIENT_BOOSTING_RESULT)


,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,roc_auc,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,HistGradientBoosting,HistGradientBoostingClassifier,False,True,True,0.6581,0.6114,0.6047,0.6656,0.6797,...,0.4595,0.6996,0.0214,0.6550,0.6401,0.0414,60,23,17,17


# Model 5: BalancedBagging

In [28]:
BALANCED_BAGGING_RESULT = pd.DataFrame(
    [
        build_model_row(
            model_name="BalancedBagging",
            estimator=build_balanced_bagging_estimator(random_state=RANDOM_STATE),
            scale_features=False,
            use_smote=False,
        )
    ]
)

display(BALANCED_BAGGING_RESULT)


,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,roc_auc,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,BalancedBagging,BalancedBaggingClassifier,False,False,True,0.6068,0.6187,0.5847,0.6249,0.6474,...,0.4889,0.6459,0.0240,0.6513,0.6147,0.0391,49,34,12,22


In [29]:
FINAL_MODEL_PERFORMANCE_SUMMARY = pd.concat(
    [
        RANDOM_FOREST_RESULT,
        LOGISTIC_REGRESSION_RESULT,
        STACKING_RESULT,
        HIST_GRADIENT_BOOSTING_RESULT,
        BALANCED_BAGGING_RESULT,
    ],
    ignore_index=True,
)

FINAL_MODEL_PERFORMANCE_SUMMARY = FINAL_MODEL_PERFORMANCE_SUMMARY.sort_values(
    by=[
        "test_balanced_accuracy",
        "test_f1_macro",
        "test_accuracy",
        "roc_auc",
        "minority_recall",
        "cv_accuracy_mean",
        "cv_gap_abs",
    ],
    ascending=[False, False, False, False, False, False, True],
).reset_index(drop=True)

FINAL_MODEL_PERFORMANCE_SUMMARY.insert(0, "model_rank", np.arange(1, len(FINAL_MODEL_PERFORMANCE_SUMMARY) + 1))
BEST_MODEL = FINAL_MODEL_PERFORMANCE_SUMMARY.head(1).reset_index(drop=True)


In [30]:
display(
    FINAL_MODEL_PERFORMANCE_SUMMARY[
        [
            "model_rank",
            "model_name",
            "estimator_class",
            "scale_features",
            "used_smote",
            "used_imbalanced_learn",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_f1_macro",
            "test_f1_weighted",
            "roc_auc",
            "minority_precision",
            "minority_recall",
            "minority_f1",
            "cv_accuracy_mean",
            "cv_accuracy_std",
            "cv_balanced_accuracy_mean",
            "cv_f1_macro_mean",
            "cv_gap_abs",
            "tn",
            "fp",
            "fn",
            "tp",
        ]
    ]
)
display(BEST_MODEL)


,model_rank,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,1,LogisticRegression,LogisticRegression,True,False,True,0.6325,0.6976,0.6255,0.6469,...,0.5743,0.6459,0.0192,0.7027,0.6347,0.0134,45,38,5,29
1,2,BalancedBagging,BalancedBaggingClassifier,False,False,True,0.6068,0.6187,0.5847,0.6249,...,0.4889,0.6459,0.0240,0.6513,0.6147,0.0391,49,34,12,22
2,3,HistGradientBoosting,HistGradientBoostingClassifier,False,True,True,0.6581,0.6114,0.6047,0.6656,...,0.4595,0.6996,0.0214,0.6550,0.6401,0.0414,60,23,17,17
3,4,RandomForest,RandomForestClassifier,False,True,True,0.6325,0.6107,0.5940,0.6464,...,0.4691,0.6781,0.0096,0.6217,0.6130,0.0456,55,28,15,19
4,5,Stacking_LR_RF,StackingClassifier,True,True,True,0.6410,0.6081,0.5962,0.6525,...,0.4615,0.6995,0.0140,0.6525,0.6415,0.0585,57,26,16,18


,model_rank,model_name,estimator_class,scale_features,used_smote,used_imbalanced_learn,test_accuracy,test_balanced_accuracy,test_f1_macro,test_f1_weighted,...,minority_f1,cv_accuracy_mean,cv_accuracy_std,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_gap_abs,tn,fp,fn,tp
0,1,LogisticRegression,LogisticRegression,True,False,True,0.6325,0.6976,0.6255,0.6469,...,0.5743,0.6459,0.0192,0.7027,0.6347,0.0134,45,38,5,29


In [31]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "html",
        str(NOTEBOOK_PATH),
        "--output",
        HTML_OUTPUT_STEM,
        "--output-dir",
        str(REPORTS_DIR),
    ],
    check=True,
)

print(REPORTS_DIR / f"{HTML_OUTPUT_STEM}.html")


[NbConvertApp] Converting notebook /Users/samyabrataroy/Downloads/Spring_Internship_2026/scripts/05_model_building_pipeline.ipynb to html


/Users/samyabrataroy/Downloads/Spring_Internship_2026/produced_reports/05_model_building_pipeline_rawdata.html


[NbConvertApp] Writing 364644 bytes to /Users/samyabrataroy/Downloads/Spring_Internship_2026/produced_reports/05_model_building_pipeline_rawdata.html
